# 04 · Engagement Score & Genre Rankings
A composite **engagement score** balances *quality* (rating) and *reach* (number of ratings), then **window functions** rank apps inside each genre.

$$\text{engagement} = 0.6 \cdot \frac{\text{rating}}{5} + 0.4 \cdot \frac{\log(1+\text{rating\_count})}{\max \log(1+\text{rating\_count})}$$

Log-scaling stops a few blockbuster apps from dominating; both components are normalised to 0–1 so the weights are interpretable.

In [ ]:
# ── Environment & Spark session (shared by every notebook) ─────────────────────
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/app-store-analytics-spark"
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

# Raw CSV location (override with the APPSTORE_CSV environment variable)
DATA_PATH = os.environ.get("APPSTORE_CSV", os.path.join(PROJECT_DIR, "data", "raw", "apple_appstore_apps.csv"))
PROCESSED_PATH = os.path.join(PROJECT_DIR, "data", "processed", "apps_processed.parquet")
FIG_DIR = os.path.join(PROJECT_DIR, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (SparkSession.builder
         .appName("AppStoreAnalytics")
         .config("spark.sql.adaptive.enabled", "true")                     # Adaptive Query Execution
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")  # merge small shuffle partitions
         .config("spark.sql.adaptive.skewJoin.enabled", "true")            # split skewed partitions
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | master={spark.sparkContext.master} | Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches="tight")

In [ ]:
df = spark.read.parquet(PROCESSED_PATH).cache()
print(f"Loaded {df.count():,} processed apps from {PROCESSED_PATH}")

## 1. Compute the score

In [ ]:
W_QUALITY, W_REACH = 0.6, 0.4
max_log = df.agg(F.max(F.log1p("rating_count"))).first()[0]

df_eng = (df.filter("is_rated")
            .withColumn("quality", F.col("user_rating") / 5)
            .withColumn("reach", F.log1p("rating_count") / F.lit(max_log))
            .withColumn("engagement_score", F.round(W_QUALITY * F.col("quality") + W_REACH * F.col("reach"), 4)))

df_eng.select(F.count("*").alias("rated_apps"), F.min("engagement_score").alias("min"),
              F.expr("percentile_approx(engagement_score, 0.5)").alias("median"),
              F.max("engagement_score").alias("max"), F.stddev("engagement_score").alias("sd")).show()

## 2. Top apps per genre with window functions

In [ ]:
w = Window.partitionBy("primary_genre").orderBy(F.desc("engagement_score"), F.desc("rating_count"))
df_ranked = (df_eng
    .withColumn("rank_in_genre", F.row_number().over(w))
    .withColumn("pct_rank_in_genre", F.round(F.percent_rank().over(w), 4))
    .withColumn("genre_avg_score", F.avg("engagement_score").over(Window.partitionBy("primary_genre"))))

top5 = (df_ranked.filter("rank_in_genre <= 5")
        .select("primary_genre", "rank_in_genre", "app_name", "developer", "user_rating",
                "rating_count", "price", "engagement_score")
        .orderBy("primary_genre", "rank_in_genre"))
top5.show(25, truncate=28)

In [ ]:
top10_pd = df_eng.orderBy(F.desc("engagement_score")).limit(10).select("app_name", "engagement_score").toPandas()
sample_pd = df_eng.select("engagement_score", "user_rating", "rating_count", "price_category").sample(fraction=0.2, seed=42).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.histplot(sample_pd.engagement_score, bins=40, ax=axes[0]); axes[0].set_title("Engagement score distribution")
sns.barplot(data=top10_pd, x="engagement_score", y=top10_pd.app_name.str[:28], ax=axes[1]); axes[1].set_title("Top 10 apps overall"); axes[1].set_ylabel("")
sns.boxplot(data=sample_pd, x="price_category", y="engagement_score", ax=axes[2]); axes[2].set_title("Engagement by price category")
axes[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); save(fig, "04_engagement_overview.png"); plt.show()

## 3. Sensitivity check: are rankings stable when the weights change?

In [ ]:
from scipy.stats import spearmanr
import numpy as np
sample_pd["alt_score"] = 0.4 * sample_pd.user_rating / 5 + 0.6 * np.log1p(sample_pd.rating_count) / max_log
rho, _ = spearmanr(sample_pd.engagement_score, sample_pd.alt_score)
print(f"Spearman rank correlation between 60/40 and 40/60 weightings: {rho:.3f}")

## 4. Strengths & limitations
- **Strengths:** combines quality and reach, robust to extreme rating counts, comparable across genres, simple to explain.
- **Limitations:** weights are a design choice (checked above via sensitivity analysis); no retention, revenue or recency signals; ratings can be inflated or manipulated.
- **Improvement:** learn the weights against a business outcome (e.g. revenue or retention) once such data is available.